# Benchmark plots

Reads the CSVs produced by `run_ip.sh`, `run_dt_rows.sh`, `run_dt_cols.sh`
and draws paper-ready figures, saving each as a PDF under `figures/`.

**IP and DT are kept in separate figures** — a set of `ip_*` figures for the
exact integer program and a set of `dt_*` figures for the decision-tree
approach (the two DT query types share a figure since both are DT).

Every config was run with warm-ups + several recorded repeats, so each point
is the **mean over reps** with a **±std** error bar.

## Design language (fonts, sizing, palette)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from plot_style import (apply_style, fig_width, fig_height, text_width,
                        inches_per_pt, CB_palette, cb_color_list)

custom_font = apply_style()     # design language lives in plot_style.py

## Load data + helpers

Point these paths at the CSVs your sweep scripts wrote.  Missing files are
skipped, so you can plot whichever experiments you have run so far.

In [ ]:
# series colours (distinct from the case study's): IP, cardinality, term-restricted
C_IP, C_CARD, C_TERM = CB_palette['CBpurple'], CB_palette['CBorange'], CB_palette['CBteal']

OUTDIR = 'figures'
os.makedirs(OUTDIR, exist_ok=True)

PATHS = dict(
    ip='ip_results.csv',                               # run_ip.sh
    agn_rows='dt_rule_length_agnostic_rows.csv',       # run_dt_rows.sh
    fix_rows='dt_rule_length_fixed_rows.csv',          # run_dt_rows.sh
    agn_cols='dt_rule_length_agnostic_cols.csv',       # run_dt_cols.sh
    fix_cols='dt_rule_length_fixed_cols.csv',          # run_dt_cols.sh
)

def load(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    print(f"[skip] {path} not found")
    return None

data = {k: load(p) for k, p in PATHS.items()}

def agg(df, x, y):
    """Mean and std of column y over reps, grouped by x, sorted by x."""
    g = df.groupby(x)[y].agg(['mean', 'std']).reset_index().sort_values(x)
    return g[x].to_numpy(), g['mean'].to_numpy(), g['std'].fillna(0).to_numpy()

def series(ax, df, x, y, label, color, marker='o'):
    """Plot one mean+/-std series onto ax (no-op if df is None)."""
    if df is None:
        return
    xs, m, s = agg(df, x, y)
    ax.errorbar(xs, m, yerr=s, label=label, color=color, marker=marker,
                capsize=2, elinewidth=0.8)

def save(fig, name):
    fig.savefig(f'{OUTDIR}/{name}.pdf')      # exact figure size: no bbox_inches='tight'
    print(f'saved {OUTDIR}/{name}.pdf')

## DT accuracy across all subsets (cardinality query)

For the decision-tree cardinality benchmark, one table per sweep: the varying value
(`n_rows` or `n_cols`), the number of subsets found, and the mean of each
distance metric across **all** subsets found. Needs CSVs from the current `dt_rule_length_agnostic.py`.

In [ ]:
METRICS = ["tvd", "jsd", "kld", "hellinger", "bc"]

def accuracy_table(df, x):
    """Per value of x: number of subsets and the mean of each metric across all
    subsets the cardinality query found.  The query is deterministic, so every rep
    holds the same values and the first is used."""
    if df is None:
        return None
    cols = [f"{m}_mean" for m in METRICS]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"[skip] accuracy vs {x}: CSV has no {', '.join(missing[:3])}, ... "
              f"-- re-run the cardinality benchmark")
        return None
    t = df.sort_values("rep").groupby(x, as_index=False).first()
    return t[[x, "n_subsets"] + cols].sort_values(x).reset_index(drop=True)

acc_rows = accuracy_table(data["agn_rows"], "n_rows")    # varying rows (cols fixed at 16)
acc_cols = accuracy_table(data["agn_cols"], "n_cols")    # varying columns (rows fixed at 32000)
display(acc_rows)
display(acc_cols)

# IP figures — exact integer program

The IP sweep varies rows only (columns fixed at 3, since column-wise sorting
needs a single column).

## IP Fig 1 — solve time vs rows

In [ ]:
fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout='constrained')
series(ax, data['ip'], 'n_rows', 'solve_sec', 'IP (exact)', C_IP, 's')
ax.set_xscale('log', base=2)
ax.set_xlabel('rows $n$')
ax.set_ylabel('solve time (s)')
save(fig, 'ip_runtime_vs_rows')

## IP Fig 2 — islands vs rows

In [ ]:
fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout='constrained')
series(ax, data['ip'], 'n_rows', 'islands', 'IP islands', C_IP, 's')
ax.set_xscale('log', base=2)
ax.set_xlabel('rows $n$')
ax.set_ylabel('number of islands')
save(fig, 'ip_islands_vs_rows')

# DT figures — decision-tree approach

Each DT figure overlays the two query types (cardinality and
term-restricted).  Rows sweep with columns fixed at 16; columns sweep with
rows fixed at 32000.

## DT Fig 1 — query time vs rows

In [ ]:
fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout='constrained')
series(ax, data['agn_rows'], 'n_rows', 'query_sec', 'cardinality', C_CARD, 'o')
series(ax, data['fix_rows'], 'n_rows', 'query_sec', 'term-restricted', C_TERM, '^')
ax.set_xscale('log', base=2)
ax.set_xlabel('rows $n$')
ax.set_ylabel('query time (s)')
ax.legend(frameon=False)
save(fig, 'dt_runtime_vs_rows')

## DT Fig 2 — index build time vs rows (one index, shared by both query types)

In [ ]:
fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout='constrained')
series(ax, data['agn_rows'], 'n_rows', 'build_sec', 'DT index', C_CARD, 'o')   # one index for both query types
ax.set_xscale('log', base=2)
ax.set_xlabel('rows $n$')
ax.set_ylabel('index build time (s)')
save(fig, 'dt_buildtime_vs_rows')

## DT Fig 3 — query time vs columns (rows fixed at 32000)

In [ ]:
fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout='constrained')
series(ax, data['agn_cols'], 'n_cols', 'query_sec', 'cardinality', C_CARD, 'o')
series(ax, data['fix_cols'], 'n_cols', 'query_sec', 'term-restricted', C_TERM, '^')
ax.set_xlabel('columns')
ax.set_ylabel('query time (s)')
ax.legend(frameon=False)
save(fig, 'dt_runtime_vs_cols')

## DT Fig 4 — rule length vs rows (cardinality query)

In [ ]:
fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout='constrained')
series(ax, data['agn_rows'], 'n_rows', 'rule_length', 'cardinality', C_CARD, 'o')
ax.set_xscale('log', base=2)
ax.set_xlabel('rows $n$')
ax.set_ylabel('rule length (terms)')
save(fig, 'dt_rulelength_vs_rows')

## DT Fig 5 — match quality (TVD) vs rows for the term-restricted (K-box) query

In [ ]:
fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout='constrained')
series(ax, data['fix_rows'], 'n_rows', 'tvd', 'term-restricted ($K$ boxes)', C_TERM, '^')
ax.set_xscale('log', base=2)
ax.set_xlabel('rows $n$')
ax.set_ylabel('TVD to target')
save(fig, 'dt_tvd_vs_rows')

# Combined 1×4 figure

One figure, four panels in a single row (full text width, `width_pt = 506.295`):
- (a) IP runtime vs rows
- (b) DT runtime vs rows
- (c) DT build time vs rows
- (d) DT runtime vs columns

In [ ]:
from matplotlib.lines import Line2D

# Full text width (516 pt) for the 1x4 row; + 1.05 in for the shared legend, panel
# titles, tick labels and axis labels, so the plot areas keep roughly the
# design-language aspect.
fig, axs = plt.subplots(1, 4, layout='constrained',
                        figsize=(text_width, text_width / 4 * 0.77 + 1.05))

# (a) IP runtime vs rows
ax = axs[0]
series(ax, data['ip'], 'n_rows', 'solve_sec', 'IP (exact)', C_IP, 's')
ax.set_xscale('log', base=2)
ax.set_xlabel('rows'); ax.set_ylabel('time (s)')
ax.set_title('(a) IP runtime')

# (b) DT runtime vs rows
ax = axs[1]
series(ax, data['agn_rows'], 'n_rows', 'query_sec', 'cardinality', C_CARD, 'o')
series(ax, data['fix_rows'], 'n_rows', 'query_sec', 'term-restricted', C_TERM, '^')
ax.set_xscale('log', base=2)
ax.set_xlabel('rows')
ax.set_title('(b) DT runtime')

# (c) DT index build time vs rows
ax = axs[2]
series(ax, data['agn_rows'], 'n_rows', 'build_sec', 'cardinality', C_CARD, 'o')   # one index for both query types
ax.set_xscale('log', base=2)
ax.set_xlabel('rows')
ax.set_title('(c) DT build')

# (d) DT runtime vs columns
ax = axs[3]
series(ax, data['agn_cols'], 'n_cols', 'query_sec', 'cardinality', C_CARD, 'o')
series(ax, data['fix_cols'], 'n_cols', 'query_sec', 'term-restricted', C_TERM, '^')
ax.set_xlabel('columns')
ax.set_title('(d) DT vs columns')

# one shared legend across the top (built from the series styles, so it is complete
# even when a CSV is missing)
fig.legend(handles=[Line2D([], [], color=C_IP, marker='s', label='IP (exact)'),
                    Line2D([], [], color=C_CARD, marker='o', label='cardinality'),
                    Line2D([], [], color=C_TERM, marker='^', label='term-restricted')],
           loc='outside upper center', ncol=3, frameon=False)

save(fig, 'combined_1x4')